# 🧩 Mini-Lab: Integration Tests for the CLI

**Module: Delivery: CLI, API, and Debugging Lab 2** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Distinguish** unit tests (call internal functions) from integration tests (run the shipped entry point)
2. **Write** integration tests that invoke a CLI via `subprocess` and assert on stdout, stderr, and exit code
3. **Use** test fixtures to set up and tear down temporary files and environment variables
4. **Understand** why integration tests catch bugs that unit tests miss

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Integration Test | Exercise the shipped entry point, not just internal functions, so you catch real wiring issues across components, config, and dependencies. |
| Unit Test | A small assertion about one behavior. A unit test that cannot fail is theatre, it should catch a real bug when the behavior breaks. |
| Command-Line Interface | A CLI contract defines how a command is used and how it behaves: its arguments and flags, what `--help` promises, which exit codes mean success or failure, and what goes to stdout versus stderr. |
| Test Fixture | Reusable sample data and helpers (like pytest fixtures and tmp_path) that make tests repeatable, isolated, and easy to read, often using golden-output comparison to catch regressions. |


## Unit Tests vs Integration Tests

In `mini-build-cli` we built a CLI with `argparse`. In `lab-runtime-failure-debugging` we saw that **tests can pass while the app is broken at runtime**. Why? Because most tests are **unit tests** — they call internal functions directly and never run the actual CLI command.

| | Unit Test | Integration Test |
|---|-----------|------------------|
| **What it calls** | `from core import word_count; word_count("hi")` | `subprocess.run(["python", "cli.py", "file.txt"])` |
| **Tests argparse?** | ❌ No | ✅ Yes |
| **Tests exit codes?** | ❌ No | ✅ Yes |
| **Tests stdout/stderr?** | ❌ No | ✅ Yes |
| **Catches missing env vars?** | ❌ Usually not | ✅ Yes (if fixture omits them) |
| **Speed** | Very fast | Slightly slower (subprocess startup) |

Both are valuable. Unit tests give fast feedback on logic. Integration tests verify that **the thing you actually ship** works end-to-end.

## Setup

We'll create a small CLI application, then write both unit tests and integration tests for it using `pytest`. All files are written to disk with `%%writefile` — the same workflow from `mini-build-cli`.

In [1]:
import subprocess, sys, os
from pathlib import Path

from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

In [2]:
!pip install -q pytest


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\molak\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## The Application Under Test

We'll build a tiny CLI that counts words in a file — similar to `mini-build-cli` but self-contained for this lab.

In [3]:
# Create a project directory
proj = Path("wc_app")
proj.mkdir(exist_ok=True)
print("Project directory created.")

Project directory created.


In [4]:
%%writefile wc_app/core.py
"""Core logic — pure function."""


def word_count(text: str) -> dict:
    """Count words and characters in text."""
    return {"words": len(text.split()), "chars": len(text)}

Writing wc_app/core.py


In [5]:
%%writefile wc_app/cli.py
"""CLI interface seam."""
import argparse
import json
import sys
from pathlib import Path

from core import word_count


def main():
    parser = argparse.ArgumentParser(
        description="Count words in a text file."
    )
    parser.add_argument("file", help="Path to the text file")
    parser.add_argument(
        "--json", action="store_true", dest="as_json",
        help="Output as JSON instead of plain text",
    )
    args = parser.parse_args()

    path = Path(args.file)
    if not path.exists():
        print(f"error: file not found: {path}", file=sys.stderr)
        sys.exit(1)

    text = path.read_text(encoding="utf-8")
    result = word_count(text)

    if args.as_json:
        print(json.dumps(result))
    else:
        print(result["words"])


if __name__ == "__main__":
    main()

Writing wc_app/cli.py


## Step 1 — A Unit Test (for Comparison)

A unit test calls the **core function** directly. It's fast and useful, but it never touches `argparse`, files, or exit codes.

In [6]:
%%writefile wc_app/test_unit.py
"""Unit tests — call internal functions directly."""
from core import word_count


def test_word_count_basic():
    result = word_count("hello world")
    assert result["words"] == 2
    assert result["chars"] == 11


def test_word_count_empty():
    result = word_count("")
    assert result["words"] == 0
    assert result["chars"] == 0

Writing wc_app/test_unit.py


In [7]:
# Run the unit tests
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_unit.py", "-v"],
    capture_output=True, text=True, cwd="wc_app",
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\molak\.virtualenvs\text_llm_agent\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-integration-tests\wc_app
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 2 items

test_unit.py::test_word_count_basic PASSED                               [ 50%]
test_unit.py::test_word_count_empty PASSED                               [100%]

============================== 2 passed in 0.04s ==============================



Unit tests pass ✅ — but they didn't test:
- Does `argparse` parse the `file` argument correctly?
- Does the CLI exit with code `1` when the file is missing?
- Does `--json` produce valid JSON on stdout?
- Does the error message go to **stderr** (not stdout)?

These are exactly the gaps that **integration tests** fill.

## Step 2 — Integration Tests with Fixtures

> 🧩 **Integration test.** Watch for the tests to run `python cli.py` and assert on stdout, stderr, and exit code.

An integration test runs the **shipped entry point** (`python cli.py ...`) as a subprocess and checks:
- **stdout** — the actual output the user sees
- **stderr** — error/diagnostic messages
- **exit code** — 0 for success, non-zero for failure

### Test Fixtures

A **test fixture** is setup/teardown code that creates a controlled environment for each test. In `pytest`, we use the `@pytest.fixture` decorator or built-in fixtures like `tmp_path`.

| Fixture | What it does |
|---------|-------------|
| `tmp_path` | Creates a unique temporary directory for each test — auto-cleaned |
| Custom fixture | Creates sample files, sets env vars, or prepares any test data |

Fixtures ensure each test starts from a **known state** — no leftover files from previous runs.


In [8]:
%%writefile wc_app/test_integration.py
"""Integration tests — run the shipped CLI entry point."""
import json
import subprocess
import sys
from pathlib import Path

import pytest

# Path to the CLI script (anchored to this test file's location)
CLI = str(Path(__file__).resolve().parent / "cli.py")


def run_cli(*args, cwd=None):
    """Helper: invoke the CLI and return the CompletedProcess."""
    return subprocess.run(
        [sys.executable, CLI, *args],
        capture_output=True,
        text=True,
        cwd=cwd,
    )


# ---- Fixture: create a sample text file in a temp directory ----

@pytest.fixture
def sample_file(tmp_path):
    """Create a temporary text file with known content."""
    f = tmp_path / "sample.txt"
    f.write_text("the quick brown fox jumps", encoding="utf-8")
    return f


# ---- Integration tests ----

def test_success_plain_output(sample_file):
    """CLI prints the word count to stdout and exits 0."""
    result = run_cli(str(sample_file))

    assert result.returncode == 0
    assert result.stdout.strip() == "5"


def test_success_json_output(sample_file):
    """--json flag produces valid JSON on stdout."""
    result = run_cli(str(sample_file), "--json")

    assert result.returncode == 0
    data = json.loads(result.stdout)
    assert data["words"] == 5
    assert data["chars"] == 25


def test_missing_file_exits_1():
    """Missing file → error on stderr, exit code 1, nothing on stdout."""
    result = run_cli("nonexistent.txt")

    assert result.returncode == 1
    assert "error:" in result.stderr
    assert "file not found" in result.stderr
    assert result.stdout == ""  # no output leaked to stdout


def test_missing_argument():
    """No arguments → argparse error on stderr, exit code 2."""
    result = run_cli()

    assert result.returncode == 2  # argparse convention
    assert result.stderr  # argparse writes usage to stderr


def test_runs_from_different_directory(sample_file, tmp_path):
    """CLI works even when called from a different working directory."""
    other_dir = tmp_path / "elsewhere"
    other_dir.mkdir()

    result = run_cli(str(sample_file), cwd=str(other_dir))

    assert result.returncode == 0
    assert result.stdout.strip() == "5"

Writing wc_app/test_integration.py


### What Each Test Verifies

| Test | What it checks | Why unit tests can't |
|------|---------------|---------------------|
| `test_success_plain_output` | stdout has the count, exit code is 0 | Unit tests don't run argparse or check exit codes |
| `test_success_json_output` | `--json` flag works end-to-end | Unit tests don't test argument parsing |
| `test_missing_file_exits_1` | Error goes to stderr, stdout is empty, exit 1 | Unit tests don't check stream separation |
| `test_missing_argument` | argparse exits with code 2 | Unit tests never invoke the parser |
| `test_runs_from_different_directory` | CLI works from any CWD | Unit tests run in-process with a fixed CWD |

In [9]:
# Run the integration tests
result = subprocess.run(
    [sys.executable, "-m", "pytest", "test_integration.py", "-v"],
    capture_output=True, text=True, cwd="wc_app",
)
print(result.stdout)
if result.stderr:
    print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.12.10, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\molak\.virtualenvs\text_llm_agent\Scripts\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\molak\MJ_Projects\19_atomic_ai_eng_course\1_ai_engineering_track\output\content\labs\mini-integration-tests\wc_app
plugins: anyio-4.9.0, asyncio-1.4.0
asyncio: mode=Mode.STRICT, debug=False, asyncio_default_fixture_loop_scope=None, asyncio_default_test_loop_scope=function
collecting ... collected 5 items

test_integration.py::test_success_plain_output PASSED                    [ 20%]
test_integration.py::test_success_json_output PASSED                     [ 40%]
test_integration.py::test_missing_file_exits_1 PASSED                    [ 60%]
test_integration.py::test_missing_argument PASSED                        [ 80%]
test_integration.py::test_runs_from_different_directory PASSED           [100%]

============================== 5 passed in 1

All five integration tests pass ✅

Notice the **fixture pattern**: `sample_file` uses `tmp_path` (a built-in pytest fixture) to create a temporary text file. Each test gets its own fresh temporary directory — no leftover state between tests.

## Step 3 — Anatomy of the `run_cli` Helper

The key pattern for CLI integration tests is a small helper that wraps `subprocess.run`:

```python
def run_cli(*args, cwd=None):
    return subprocess.run(
        [sys.executable, CLI, *args],
        capture_output=True,
        text=True,
        cwd=cwd,
    )
```

| Parameter | Why |
|-----------|-----|
| `sys.executable` | Uses the **same Python** that runs pytest — avoids version mismatches |
| `CLI` (absolute path) | Anchored to the script's location so it works from any `cwd` |
| `capture_output=True` | Captures stdout and stderr separately for assertions |
| `text=True` | Returns strings instead of bytes |
| `cwd=cwd` | Lets tests simulate running from a different directory |

With this helper, every test is just: **call `run_cli(...)`, then assert on `.returncode`, `.stdout`, and `.stderr`**.

## Cleanup

In [10]:
import shutil

shutil.rmtree("wc_app", ignore_errors=True)
print("Cleaned up.")

Cleaned up.


## 🎯 Summary

### Key Takeaways

1. **Unit tests call functions, integration tests run the entry point** — unit tests import and call `word_count()` directly; integration tests invoke `python cli.py` via `subprocess` and check stdout, stderr, and exit code
2. **Integration tests catch what unit tests miss** — argument parsing bugs, wrong exit codes, output going to the wrong stream, and CWD-dependent failures only surface when you run the shipped command
3. **Test fixtures create a controlled environment** — `tmp_path` gives each test its own temporary directory, and custom fixtures (like `sample_file`) set up known test data that's automatically cleaned up
4. **The `run_cli` helper is the core pattern** — wrap `subprocess.run` with `sys.executable`, an anchored script path, and `capture_output=True`, then assert on the three outputs: returncode, stdout, and stderr